# Page axes — L1 / L2 / L3 / specialty (Colab first)

One DistilBERT encoder, four heads (codeability is not learned: the pipeline reads it from the taxonomy tag of the Final page type). **Colab:** Runtime → Change runtime type → **GPU**, then edit `BASE_PATH` only.

Put these in one Drive folder (`BASE_PATH`):

* `train_axes.py`, `predict_axes.py`, `taxonomy.json` (v29.2), `requirements.txt`, `check_test_pages.py`, `test_pages.jsonl` — from `training/bert-training/`
* `training_data.jsonl` — from `Training/processed/` (2,750 pages, v29.2 labels)
* `mock_pages.jsonl` — optional, from `Training/mock/` (25 synthetic pages; with `USE_MOCK = True` they train, never score)

The model is written to `BASE_PATH/page-axes/` after every improving epoch. On a PC the same notebook runs from `training/bert-training/` with `data/` and `output/` there. Page text is never printed — only counts and metrics.

Small classes (2026-10-25): the split spreads every L1, L2, L3 and specialty over train / val / test (`--split labels`); classes with fewer than 30 training pages are oversampled (`--oversample 30`, at most 4 repeats); loss weights are softened (`--class-weight sqrt`). All three are the defaults. `split_report.csv` lists pages per side for every class.

In [ ]:
import json, os, sys, subprocess
from pathlib import Path

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/page-axes-training')  # <- the only thing to edit
    CODE, DATA, OUTPUT = BASE_PATH, BASE_PATH / 'training_data.jsonl', BASE_PATH / 'page-axes'
    MOCK = BASE_PATH / 'mock_pages.jsonl'
else:
    CODE = Path.cwd()                                              # run from training/bert-training
    DATA, OUTPUT = CODE / 'data' / 'training_data.jsonl', CODE / 'output' / 'page-axes'
    MOCK = CODE / 'data' / 'mock_pages.jsonl'
USE_MOCK = True    # True: add the synthetic pages (training split only)
EPOCHS = 20

# A Hugging Face name, or a folder holding the weights where Hugging Face is blocked.
BASE_MODEL = 'distilbert-base-uncased'

for name in ('train_axes.py', 'predict_axes.py', 'taxonomy.json', 'check_test_pages.py', 'test_pages.jsonl'):
    assert (CODE / name).is_file(), f'{name} is missing from {CODE}'
assert DATA.is_file(), f'no training data at {DATA}'
assert not USE_MOCK or MOCK.is_file(), f'USE_MOCK is on but there is no {MOCK}'
assert json.loads((CODE / 'taxonomy.json').read_text())['version'] == 'v29.2', 'taxonomy.json is not v29.2'
sys.path.insert(0, str(CODE))
print('code:', CODE, '| data:', DATA, '| output:', OUTPUT)

In [ ]:
if ON_COLAB:
    %pip install -q -r {str(CODE / 'requirements.txt')}

Which device will train: a CUDA GPU when PyTorch sees one, otherwise the CPU (slow: expect tens of minutes per epoch).

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
def train(*args):
    """Run train_axes.py with these paths; its output streams below."""
    cmd = [sys.executable, str(CODE / 'train_axes.py'), '--data', str(DATA), '--output', str(OUTPUT),
           '--taxonomy', str(CODE / 'taxonomy.json'), '--base-model', BASE_MODEL,
           *(['--with-mock', str(MOCK)] if USE_MOCK else []), *args]
    # Read the child's output line by line: a child process writes to the kernel's own stdout,
    # which Colab and Jupyter do not reliably show in the cell.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

Compatibility check: label space, split, two training steps and a minutes-per-epoch estimate. Saves nothing. Read the head counts it prints — they say how many pages train each head and why the others are left out.

In [ ]:
train('--check')

A run: 20 epochs (`EPOCHS`). The output folder keeps the best epoch by mean validation macro F1 of L1, L2 and L3. It prints the running training loss every `--log-every` steps (10), then each epoch's train loss, validation loss and validation macro F1 per head. At the end: per-class L1 recall and the **Lab Report <-> Progress Note** confusion, both ways. Options: `--epochs`, `--log-every`, `--min-l1-pages` (11: an L1 needs more than 10 pages), `--test-fraction` (0.15), `--select-heads`.

In [ ]:
train('--epochs', str(EPOCHS))

In [ ]:
import json
import pandas as pd
pd.read_csv(OUTPUT / 'training_log.csv')

Scored on the held-out **test** groups (charts and stand-alone pages), with the best epoch loaded back from disk. Full-path accuracy is what the client experiences; hierarchical P/R gives partial credit when L1 is right and L2 wrong.

In [ ]:
summary = json.loads((OUTPUT / 'metrics_summary.json').read_text())
print(json.dumps(summary, indent=2))

In [ ]:
metrics = pd.read_csv(OUTPUT / 'category_wise_metrics.csv')
metrics[metrics.support > 0].sort_values(['head', 'f1'])

Small classes: L1 recall for the ones with few pages (the addendum's check: Administrative, Provider Letter, Problem List, Immunization), and pages per side for every class.

In [ ]:
small = ['Administrative', 'Provider Letter', 'Problem List', 'Immunization']
print(metrics[(metrics['head'] == 'L1') & metrics['class'].isin(small)][['class', 'recall', 'support', 'train_pages']])
split = pd.read_csv(OUTPUT / 'split_report.csv')
split[split.train < 30].sort_values(['head', 'train'])

In [ ]:
pd.read_csv(OUTPUT / 'confusion' / 'L1.csv', index_col=0)

In [ ]:
# Within one L1: L2 truth against L2 predicted (given the right L1).
print(sorted(p.name for p in (OUTPUT / 'confusion').glob('L2_*.csv')))
pd.read_csv(OUTPUT / 'confusion' / 'L2_lab_report.csv', index_col=0)

Errors (no text): `route` says where the pipeline would send the page — `model`, `canon` (keyword canon decides) or `review`. Confident errors (route `model`) are the ones to look at first; they are often label errors to fix in the annotation tool.

In [ ]:
errors = pd.read_csv(OUTPUT / 'errors.csv')
print(errors.route.value_counts().to_dict())
errors[errors.route == 'model'].sort_values('L1_confidence', ascending=False).head(30)

## Synthetic test pages

Twelve made-up pages (`test_pages.jsonl`) with the expected answer. PASS/FAIL per head; the cell fails if any Lab Report page comes back as Progress Note.

In [ ]:
result = subprocess.run([sys.executable, str(CODE / 'check_test_pages.py'), '--model', str(OUTPUT),
                         '--pages', str(CODE / 'test_pages.jsonl')], cwd=CODE, capture_output=True, text=True)
print(result.stdout, result.stderr)
assert result.returncode == 0, 'a Lab Report test page came back as Progress Note'

## Inference check

Paste a page's text. L2 is picked among the predicted L1's L2s only, L3 among the L2's L3s.

In [ ]:
from predict_axes import AxesClassifier
clf = AxesClassifier(OUTPUT)
print('L1 classes:', clf.space['L1'])

In [ ]:
import ipywidgets as widgets
from IPython.display import display

box = widgets.Textarea(placeholder='Paste page text here', layout=widgets.Layout(width='100%', height='220px'))
button = widgets.Button(description='Classify', button_style='primary')
result = widgets.Output()

def on_click(_):
    with result:
        result.clear_output()
        clf.show(box.value)

button.on_click(on_click)
display(box, button, result)

In [ ]:
# No widgets? Paste the text between the triple quotes and run this cell.
TEXT = """
paste page text here
"""
clf.show(TEXT)